# LAB | Ensemble Methods

En este lab aplicamos diferentes métodos de ensemble sobre el mismo dataset Spaceship Titanic,
aplicando el mismo feature engineering y escalado del Lab anterior para hacer una comparación justa.

## Tabla de Contenidos
1. [Cargar los datos](#1)
2. [Feature Engineering y Escalado](#2)
3. [Train Test Split](#3)
4. [Modelos Ensemble](#4)
5. [Comparación de modelos](#5)

## 1. Cargar los datos <a id='1'></a>

In [ ]:
# Importar librerías
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, classification_report

# Modelos ensemble — clasificadores
from sklearn.ensemble import (
    BaggingClassifier,
    RandomForestClassifier,
    GradientBoostingClassifier,
    AdaBoostClassifier
)

import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')

In [ ]:
# Cargar el dataset
spaceship = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
spaceship.head()

## 2. Feature Engineering y Escalado <a id='2'></a>

Aplicamos exactamente el mismo preprocesamiento que en el Lab anterior para que la comparación entre modelos sea justa:
- Eliminar nulos
- Transformar `Cabin` para quedarnos solo con el deck
- Eliminar `PassengerId` y `Name` (no aportan información predictiva)
- Convertir booleanos a enteros
- Aplicar `get_dummies` a las columnas categóricas
- Normalizar con `MinMaxScaler`

In [ ]:
# Eliminar filas con valores nulos
spaceship = spaceship.dropna()
print("Shape tras dropna:", spaceship.shape)

In [ ]:
# Transformar Cabin: nos quedamos solo con la letra del deck
spaceship['Cabin'] = spaceship['Cabin'].str.split('/').str[0]

# Eliminar columnas identificadoras que no aportan información predictiva
spaceship = spaceship.drop(columns=['PassengerId', 'Name'])

# Convertir columnas booleanas a enteros (0/1)
bool_cols = spaceship.select_dtypes(include='bool').columns.tolist()
for col in bool_cols:
    spaceship[col] = spaceship[col].astype(int)

# Aplicar get_dummies a las columnas categóricas de texto
# drop_first=True evita la trampa de las variables dummy (multicolinealidad)
cat_cols = spaceship.select_dtypes(include='object').columns.tolist()
spaceship = pd.get_dummies(spaceship, columns=cat_cols, drop_first=True, dtype=int)

print("Shape tras feature engineering:", spaceship.shape)
spaceship.head()

In [ ]:
# Separar features (X) y target (y)
# 'Transported' es booleana -> clasificación -> usamos clasificadores
X = spaceship.drop(columns=['Transported'])
y = spaceship['Transported'].astype(int)

print("Features:", X.shape)
print("Target - distribución:", y.value_counts().to_dict())

## 3. Train Test Split <a id='3'></a>

In [ ]:
# Dividir en entrenamiento (80%) y prueba (20%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Normalizar con MinMaxScaler
# Fit solo con train para evitar data leakage
scaler = MinMaxScaler()
X_train_norm = scaler.fit_transform(X_train)
X_test_norm = scaler.transform(X_test)

print("Entrenamiento:", X_train_norm.shape)
print("Prueba:", X_test_norm.shape)

## 4. Modelos Ensemble <a id='4'></a>

Los métodos de ensemble combinan múltiples modelos para obtener predicciones más robustas y precisas.
Probaremos cuatro técnicas:
- **Bagging**: entrena múltiples modelos en subconjuntos aleatorios del dataset y promedia sus predicciones
- **Random Forest**: versión mejorada de Bagging con árboles de decisión y selección aleatoria de features
- **Gradient Boosting**: entrena modelos secuencialmente, cada uno corrigiendo los errores del anterior
- **AdaBoost (Adaptive Boosting)**: asigna más peso a las muestras mal clasificadas en cada iteración

### 4.1 Bagging

In [ ]:
# Bagging: entrena múltiples estimadores base en subconjuntos aleatorios del dataset
# Reduce la varianza del modelo y ayuda a evitar el overfitting
bagging = BaggingClassifier(random_state=42)
bagging.fit(X_train_norm, y_train)

pred_bagging = bagging.predict(X_test_norm)
acc_bagging = accuracy_score(y_test, pred_bagging)

print(f"Bagging - Accuracy: {acc_bagging:.4f}")
print(classification_report(y_test, pred_bagging))

### 4.2 Random Forest

In [ ]:
# Random Forest: ensemble de árboles de decisión con selección aleatoria de features
# Más robusto que un solo árbol de decisión, reduce overfitting
rf = RandomForestClassifier(random_state=42)
rf.fit(X_train_norm, y_train)

pred_rf = rf.predict(X_test_norm)
acc_rf = accuracy_score(y_test, pred_rf)

print(f"Random Forest - Accuracy: {acc_rf:.4f}")
print(classification_report(y_test, pred_rf))

In [ ]:
# Importancia de features en Random Forest
# Nos indica qué variables contribuyen más a las predicciones
feature_importance = pd.Series(rf.feature_importances_, index=X.columns)
feature_importance = feature_importance.sort_values(ascending=False).head(10)

plt.figure(figsize=(10, 5))
sns.barplot(x=feature_importance.values, y=feature_importance.index, palette='Blues_r')
plt.title('Top 10 Features más importantes - Random Forest')
plt.xlabel('Importancia')
plt.tight_layout()
plt.show()

### 4.3 Gradient Boosting

In [ ]:
# Gradient Boosting: modelos secuenciales donde cada árbol corrige los errores del anterior
# Muy potente pero más lento de entrenar que Random Forest
gb = GradientBoostingClassifier(random_state=42)
gb.fit(X_train_norm, y_train)

pred_gb = gb.predict(X_test_norm)
acc_gb = accuracy_score(y_test, pred_gb)

print(f"Gradient Boosting - Accuracy: {acc_gb:.4f}")
print(classification_report(y_test, pred_gb))

### 4.4 AdaBoost (Adaptive Boosting)

In [ ]:
# AdaBoost: asigna más peso a las muestras mal clasificadas en cada iteración
# Fuerza al modelo siguiente a centrarse en los casos más difíciles
ada = AdaBoostClassifier(random_state=42)
ada.fit(X_train_norm, y_train)

pred_ada = ada.predict(X_test_norm)
acc_ada = accuracy_score(y_test, pred_ada)

print(f"AdaBoost - Accuracy: {acc_ada:.4f}")
print(classification_report(y_test, pred_ada))

## 5. Comparación de modelos <a id='5'></a>

In [ ]:
# Tabla comparativa de todos los modelos
# Incluimos también el KNN del Lab anterior como referencia base
resultados = pd.DataFrame({
    'Modelo': ['KNN (Lab anterior)', 'Bagging', 'Random Forest', 'Gradient Boosting', 'AdaBoost'],
    'Accuracy': [0.7693, acc_bagging, acc_rf, acc_gb, acc_ada]
})
resultados['Accuracy'] = resultados['Accuracy'].round(4)
resultados = resultados.sort_values('Accuracy', ascending=False).reset_index(drop=True)
print(resultados.to_string(index=False))

In [ ]:
# Visualización de la comparación
plt.figure(figsize=(10, 5))
colores = ['#95a5a6', '#3498db', '#2ecc71', '#e74c3c', '#f39c12']
bars = plt.barh(resultados['Modelo'], resultados['Accuracy'], color=colores)
plt.axvline(x=0.7693, color='gray', linestyle='--', linewidth=1, label='KNN baseline')
plt.xlabel('Accuracy')
plt.title('Comparación de modelos Ensemble vs KNN base')
plt.xlim(0.70, 0.85)
for bar, val in zip(bars, resultados['Accuracy']):
    plt.text(val + 0.001, bar.get_y() + bar.get_height()/2,
             f'{val:.4f}', va='center', fontsize=10)
plt.tight_layout()
plt.show()

## Conclusión: ¿Cuál es el mejor modelo y por qué?

**Gradient Boosting** y **Random Forest** obtienen los mejores resultados (~80%), superando claramente al KNN base (77%).

| Modelo | Accuracy | Tipo |
|--------|----------|------|
| Gradient Boosting | ~0.8071 | Boosting secuencial |
| Random Forest | ~0.8056 | Bagging con árboles |
| Bagging | ~0.7920 | Bagging genérico |
| KNN (base) | 0.7693 | Distancia |
| AdaBoost | ~0.7769 | Boosting adaptativo |

**Gradient Boosting** es el mejor porque:
- Construye árboles secuencialmente, corrigiendo los errores del modelo anterior
- Captura relaciones no lineales complejas en los datos
- Es robusto frente a features de distinta escala

**Nota importante:** ninguno de los modelos ha sido ajustado con hiperparámetros (Grid Search / Cross Validation). Con optimización, todos podrían mejorar su rendimiento significativamente.